# Report Generator

## Initializing

In [ ]:
import datetime;import pandas as pd;import numpy as np
from os import listdir;import os, time, datetime;import glob
import matplotlib.pyplot as plt;import docx
import matplotlib;from IPython import get_ipython;from docx import Document
from docx.shared import Inches, Pt;from io import BytesIO

pd.set_option('display.max_colwidth', None)

## Test Information

In [ ]:
files_location = r"example data/"
prepared_on = datetime.datetime.now();
prepared_on = prepared_on.strftime('%m-%d-%y %H:%M:%S')
system_vin = 'System_143'
stack_vin = 'Stack_2050009'
stack_number = '18'
test_type = 'System Commissioning, Durability, Operational and Research Investigations'
FCU_software = 'FCUv9.exe'
operator_software = 'LabView_system_2.3.11'
station = 'Santa Fe'
station_stack_system_notes = 'sys24 tests'+\
' No Further Notes.'

operator_notes = 'Testing completed by Jerren Grimes, Engineer1 LastName and FirstName Engineer 2'
document_description = 'This document describes the end-of-line (EOL) test(s) associated with the stack '+ str(stack_vin) + ' and system ' + str(system_vin) + ' current up to the date shown here: '+ str(prepared_on)
server_location = 'https://classified_data.com'

## Analysis Functions

In [ ]:
def analyze_data(dfnew):
    Name = ['Operation Time (Power > 25 kW)','Maximum Net Power','Max Humidity','Min Humidity']
    Value = [str(sum(dfnew['Step Time']))+' seconds',str(round(np.max(dfnew['Output Power (kW)']-dfnew['Power Consumption (kW)']),2)) + ' kW',str(np.max(dfnew['Air Humidity (%)'])) + ' %',str(np.min(dfnew['Air Humidity (%)']))+' %']
    dict = {'Name': Name, 'Value': Value} 
    df9 = pd.DataFrame(dict)
    return df9

csv_list = files_location+str("/*.csv");my_csv_list = (glob.glob(csv_list))
txt_list = files_location+str("/*.txt");my_txt_list = (glob.glob(txt_list))
photo_list_png = files_location+str("/*.png");my_photo_list_png = (glob.glob(photo_list_png))
photo_list_jpg = files_location+str("/*.jpg");my_photo_list_jpg = (glob.glob(photo_list_jpg))

file_number = np.arange(1, len(my_csv_list)+1, 1).tolist()
full_txt_list = [None] * len(my_csv_list)
lines = [None]*len(my_csv_list)
for j in range(len(my_txt_list)):
    for i in range(len(my_csv_list)):
        if my_txt_list[j][:-3] == my_csv_list[i][:-3]:
            with open(my_txt_list[j]) as f:
                lines[i] = f.readline().strip()

file_list = [None] * len(my_csv_list)
for i in range(len(my_csv_list)):
    file_list[i] = format(os.path.basename(my_csv_list[i]))
    full_txt_list[i] = datetime.datetime.fromtimestamp(os.path.getmtime(my_csv_list[i]))


dict = {'File Number': file_number, 'File Location': my_csv_list, 'File Name': file_list, 'Created On': full_txt_list,'Notes Associated with File': lines} 
df = pd.DataFrame(dict)
df22 = df.drop('File Location', axis = 1)
for i in range(len(my_csv_list)):
    header = "DATA FILE {0} OVERVIEW".format(df.iloc[i,0])
    filename_subheader = "FILE NAME =  {0}".format(os.path.basename(my_csv_list[i]))
    filenotes_subheader = "FILE NOTES = {0}".format(df.iloc[i,3])
    creation_date = datetime.datetime.fromtimestamp(os.path.getmtime(my_csv_list[i]));


def cvm_plotter(dfnew):

        mem2file = BytesIO()
        #rows = 221
        df1 = dfnew.loc[:,:'Time[s]']
        df2 = dfnew.loc[:,'V1 (mV)':]
        df_merged = pd.concat([df1, df2], axis=1, sort=False);df_merged = df_merged.reindex(sorted(df_merged.columns), axis=1)
        matplotlib.rcParams['font.family'] = 'Times New Roman'


        plt.rcParams["axes.prop_cycle"] = plt.cycler("color", ['c', 'm', 'gold', 'k', 'r', 'g', 'b', 'deeppink','mediumspringgreen','chocolate'])
        plt.rc('font', size=15) #controls default text size
        plt.rc('axes', titlesize=15) #fontsize of the title
        plt.rc('axes', labelsize=15) #fontsize of the x and y labels
        plt.rc('xtick', labelsize=15) #fontsize of the x tick labels
        plt.rc('ytick', labelsize=15) #fontsize of the y tick labels
        #plt.rc('legend', fontsize=15) #fontsize of the legend
        
        fig, axs = plt.subplots(1,1,figsize=(15,7))
        xlim = 1.02*(df_merged['Time[s]'].max())
        plt.subplot(1,1,1)
        plt.plot(df_merged['Time[s]'],df_merged.iloc[:,2:222],label = df_merged.columns[2:222], linewidth=1)
        plt.xlabel('Time (s)')
        plt.ylabel('Voltage (V)')
        plt.ylim(1000, 2000)
        plt.grid(linestyle='dotted')
        plt.savefig(mem2file)
        plt.close()
        return mem2file


def variable_plot(dfnew,xx,yy,xlim,ylim,color,slice,plot_type):

        mem2file = BytesIO()
        matplotlib.rcParams['font.family'] = 'Times New Roman'
        plt.rc('font', size=15); plt.rc('axes', titlesize=15); plt.rc('axes', labelsize=15); plt.rc('xtick', labelsize=15); plt.rc('ytick', labelsize=15); #plt.rc('legend', fontsize=15) 
        
        fig, axs = plt.subplots(1,1,figsize=(15,7))
        if plot_type == 'scatter':
            plt.scatter(dfnew[xx],dfnew[yy], s = 8, c = color)
        elif plot_type == 'line':
            plt.plot(dfnew[xx],dfnew[yy], linewidth = 1, c = color)
        plt.xlabel(xx);plt.ylabel(yy)
        if isinstance(xlim, str) != True:
            plt.xlim(0,xlim)
        if isinstance(ylim, str) != True:
            plt.ylim(0,ylim)

        plt.title(yy[slice:])
        plt.grid(linestyle='dotted')
        #plt.show()        
        plt.savefig(mem2file)
        plt.close()
        return mem2file


def variable_doubleplot(dfnew,xx,yy1,yy2,xlim,ylim1,ylim2,c1,c2,slice1,slice2,plot_type):
        mem2file = BytesIO()
        matplotlib.rcParams['font.family'] = 'Times New Roman'
        plt.rc('font', size=15); plt.rc('axes', titlesize=15); plt.rc('axes', labelsize=15); plt.rc('xtick', labelsize=15); plt.rc('ytick', labelsize=15); #plt.rc('legend', fontsize=15) 
        samy = dfnew['Output Power (kW)']-dfnew['Power Consumption (kW)']
        samy = samy.to_list()
        dfnew.insert(200, 'Output Power (kW)',samy, True)
        fig, ax = plt.subplots(1,1,figsize=(15,7))
        
        if plot_type == 'scatter':
            ax.scatter(dfnew[xx],dfnew[yy1], s = 8, c = c1, label = yy1[slice1:])
            ax2 = ax.twinx(); ax2.scatter(dfnew[xx],dfnew[yy2], s = 8, c = c2, label = yy2[slice2:])
        elif plot_type == 'line':
            ax.plot(dfnew[xx],dfnew[yy1], linewidth = 1, c = c1, label = yy1[slice1:])
            ax2 = ax.twinx(); ax2.plot(dfnew[xx],dfnew[yy2], linewidth = 1, c = c2, label = yy2[slice2:])
    
        ax.set_xlabel(xx);ax.set_ylabel(yy1)
        ax2.set_ylabel(yy2)
        if isinstance(xlim, str) != True:
            ax.set_xlim(0,xlim)
        if isinstance(ylim1, str) != True:
            ax.set_ylim(0,ylim1)
        if isinstance(ylim2, str) != True:
            ax2.set_ylim(0,ylim2)

        ax.grid(linestyle='dotted')
        #fig.legend(loc="upper right")
        plt.savefig(mem2file); plt.close()
        return mem2file
         
def power_boxplot(df):
 
        mem2file = BytesIO()
        matplotlib.rcParams['font.family'] = 'Times New Roman'
        list = [200, 180, 160, 140, 120, 100, 80, 60, 40]
        list2 = len(list)*[0]
        j = 0
        for i in list:
    
            new = df[df['Power SP (kW)'] == i]
            new_array = np.array(new['Output Power (kW)'])
            list2[j] = new_array
            j = j + 1
    
        all_data = list2
        labels = [str(r) for r in list]

        #MultipleBoxplot
        fig = plt.subplots(1,1,figsize=(15,7))
        plt.boxplot(all_data, vert=True, patch_artist=True, tick_labels=labels) 
        plt.ylabel('Voltage (V)')
        plt.xlabel('Power (kW)')
        plt.title('Voltage vs Power')
        #plt.show()
        plt.savefig(mem2file)
        plt.close()
        return mem2file

def pol_curve(dfnew):

        mem2file = BytesIO()
        matplotlib.rcParams['font.family'] = 'Times New Roman'

        plt.rcParams["axes.prop_cycle"] = plt.cycler("color", ['r', 'm', 'gold', 'k', 'c', 'g', 'b', 'deeppink','mediumspringgreen','chocolate'])
        plt.rc('font', size=15) #controls default text size
        plt.rc('axes', titlesize=15) #fontsize of the title
        plt.rc('axes', labelsize=15) #fontsize of the x and y labels
        plt.rc('xtick', labelsize=15) #fontsize of the x tick labels
        plt.rc('ytick', labelsize=15) #fontsize of the y tick labels
        #plt.rc('legend', fontsize=15) #fontsize of the legend
        
        fig, axs = plt.subplots(1,1,figsize=(15,7))
        xx = 'Current (A)'
        yy = 'Voltage (V)'
        plt.subplot(1,1,1)
        plt.subplots_adjust(left=0.1, right=0.7, bottom=0.2, top=0.8)
        plt.scatter(df_raw_data[xx],df_raw_data[yy], s = 8)
        plt.xlabel(xx); plt.ylabel(yy)
        plt.grid(linestyle='dotted')
        plt.savefig(mem2file); plt.close()
        return mem2file

def file_data_analyzer(list_of_csvs):

    for i in range(len(list_of_csvs)):
        header = "DATA FILE {0} OVERVIEW".format(i); document.add_heading(header,0)
        
        print('File Location'); print(df.iloc[i,1]); print('Notes')
        print(df.iloc[i,4])
        df_raw_data = pd.read_csv(list_of_csvs[i],encoding='windows-1252')
        
        df3 = overview_analyzer(df_raw_data)
        
        t = document.add_table(df2.shape[0]+1, df3.shape[1])
        t.style = 'TableGrid';

        for j in range(df3.shape[-1]):
            t.cell(0,j).text = df3.columns[j]

        for i in range(df3.shape[0]):
            for j in range(df3.shape[-1]):
                t.cell(i+1,j).text = str(df3.values[i,j])


def add_df_as_table(df2):
    t = document.add_table(df2.shape[0]+1, df2.shape[1])
    t.style = 'Table Grid';

    for j in range(df2.shape[-1]):
        t.cell(0,j).paragraphs[0].add_run(df2.columns[j]).bold=True

    for i in range(df2.shape[0]):
        for j in range(df2.shape[-1]):
            t.cell(i+1,j).text = str(df2.values[i,j])

variable_list = ['Fusion Temperature [C]','Air Humidity (%)','Stoich','Inlet Pressure (bar)','Efficiency (%)','System Efficiency (%)','Nitrogen Pressure (bar)','Chlorine Pressure (bar)','Secondary Pressure (bar)','Water SP (%)','Injector SP (%)','Injector Frequency (Hz)','H2O Outlet Pressure (bar)','H2O Inlet Pressure (bar)','Voltage (V)','Current (A)','Exhaust Temperature [C]','Exhaust Temperature 2 [C]',]

def slice_by_set_power(df,slice):
    
    new = df[df['Power SP (kW)'] == slice]
    
    return new
                

def variable_statistics(df,variable_name):

    max = df[variable_name].max()
    min = df[variable_name].min()
    range = max - min
    mean = df[variable_name].mean()        
        
    if len(df[variable_name]) == 0:    
        median = np.nan
    else:
        median = df[variable_name].median()

    if df[variable_name].mode().empty == False:
        mode = df[variable_name].mode()[0]
    else:
        mode = np.nan
        
    std_dev = df[variable_name].std()
    step_time = df['Step Time'].sum()
    tuple = max, min, range, mean, median, mode, std_dev, step_time

    return [round(m,2) for m in tuple]
   

def df_statistics(df,name):

    headers = ['Power',200,180,160,140,120,100,80,60,40,30]
    headers2 = [str(r) for r in headers]
    df_statistics = pd.DataFrame(columns=headers2)
    df_statistics['Power'] = ['max','min','range','mean','median','mode','std dev', 'time spent (s)']

    for i in headers2[1:]:

        aa = slice_by_set_power(df,int(i))
        bb = variable_statistics(aa,name)
        df_statistics[i] = [*bb]

    return df_statistics 
    
    
def variable_statistics_cell(df):

    df1 = df.loc[:,:'Time[s]']
    df2 = df.loc[:,'V1 (mV)':]
    df_merged = pd.concat([df1, df2], axis=1, sort=False)
    df_merged = df_merged.reindex(sorted(df_merged.columns), axis=1)
    pd.set_option('display.max_columns', None)
    df3 = df_merged
    
    max = df3.loc[:,'V1 (mV)':'V220 (mV)'].max().max()
    min = df3.loc[:,'V1 (mV)':'V220 (mV)'].min().min()
    range = max - min
    mean = df3.loc[:,'V1 (mV)':'V220 (mV)'].mean().mean()

    if df3.loc[:,'V1 (mV)':'V220 (mV)'].empty:
        median = np.nan
        
    else:
        median = df3.loc[:,'V1 (mV)':'V220 (mV)'].median().median()

    tuple = max, min, range, mean, median#, std_dev
    
    return [round(m,2) for m in tuple]


def cvm_statistics(df):
    headers = ['Power',200,180,160,140,120,100,80,60,40,30]
    headers2 = [str(r) for r in headers]
    df_statistics_cell = pd.DataFrame(columns=headers2);

    df_statistics_cell['Power'] = ['max','min','range','mean','median']

    for i in headers2[1:]:
        
        aa = slice_by_set_power(df,int(i))
        bb = variable_statistics_cell(aa)

        df_statistics_cell[i] = [*bb]


    return df_statistics_cell


dfc1 = [];

def all_data_dataframe():
    #c = 0
    total_time = 0
    for i in my_csv_list:
        #c = c + 1
        dfc1 = pd.read_csv(i,encoding='windows-1252')
        dfc1 = dfc1.append(dfc1, ignore_index = True)

## Generate Document

In [ ]:
#%%capture
prepared_on = datetime.datetime.now();prepared_on = prepared_on.strftime('%m-%d-%y %H:%M:%S')


document = Document(r'Report Template.docx')
document.add_picture(r'Fuel Cell Report Cover.png', width=Inches(6.5))
#document.add_heading('End-of-Line Report',0)

style = document.styles['Normal'];font = style.font;font.name = 'Times New Roman';font.size = Pt(20)
p = document.add_paragraph()
p.style = document.styles['Normal']


####################PAGE 1#######################
p = document.add_paragraph('')
p.paragraph_format.space_before = Pt(340)
p = document.add_paragraph('')
p.style = document.styles['Normal']

p.add_run('SYSTEM VIN').bold = True
run = p.add_run(' = ' + system_vin)
p = document.add_paragraph('')
p.style = document.styles['Normal']
p.add_run('STACK VIN ').bold = True
run = p.add_run(' = ' + stack_vin)

style = document.styles['Normal'];font = style.font;font.name = 'Times New Roman';font.size = Pt(20)
q = document.add_paragraph()
q.style = document.styles['Normal']
q.add_run('Stack #'+stack_number).italic = True
q.alignment = 1
q = document.add_paragraph('')
q.add_run('Prepared on: '+prepared_on)
q.alignment = 1
document.add_page_break()

###################PAGE 2########################
d = document.add_paragraph()
style = document.styles['Normal'];font = style.font;font.name = 'Times New Roman';font.size = Pt(10)
d.style = document.styles['Normal']
d = document.add_paragraph('')
d.paragraph_format.space_after = Pt(250)
d = document.add_paragraph('')
d.add_run('This report was generated with the aid of a custom python notebook titled ‘Report Generator.ipynb’. For internal use only.').italic = True
d.alignment = 1
document.add_page_break()

###############PAGE 3###########################
#document.add_heading('TEST INFORMATION',0)
m = document.add_paragraph()
#style = document.styles['Normal'];font = style.font;font.name = 'Times New Roman';font.size = Pt(12)
m.style = document.styles['Normal']

d = document.add_paragraph('')
#d = document.add_paragraph('')
d.add_run('Document Description').bold = True
table = document.add_table(rows=1, cols=1);table.style = 'Table Grid';cell = table.cell(0, 0)
cell.text = document_description

d = document.add_paragraph('');d = document.add_paragraph('')
d.add_run('Test Type').bold = True
table = document.add_table(rows=1, cols=1);table.style = 'Table Grid';cell = table.cell(0, 0)
cell.text = test_type

d = document.add_paragraph('');d = document.add_paragraph('')
d.add_run('FCU Software(s)').bold = True
table = document.add_table(rows=1, cols=1);table.style = 'Table Grid';cell = table.cell(0, 0)
cell.text = FCU_software

d = document.add_paragraph('');d = document.add_paragraph('')
d.add_run('Operator Software(s)').bold = True
table = document.add_table(rows=1, cols=1);table.style = 'Table Grid';cell = table.cell(0, 0)
cell.text = operator_software

d = document.add_paragraph('');d = document.add_paragraph('')
d.add_run('Station').bold = True
table = document.add_table(rows=1, cols=1);table.style = 'Table Grid';cell = table.cell(0, 0)
cell.text = station

d = document.add_paragraph('');d = document.add_paragraph('')
d.add_run('Station/Stack/System Notes').bold = True
table = document.add_table(rows=1, cols=1);table.style = 'Table Grid';cell = table.cell(0, 0)
cell.text = station_stack_system_notes

d = document.add_paragraph('');d = document.add_paragraph('')
d.add_run('Operator Notes').bold = True
table = document.add_table(rows=1, cols=1);table.style = 'Table Grid';cell = table.cell(0, 0)
cell.text = operator_notes
document.add_page_break()

###################PAGE 4#########################
#document.add_heading('DATA FILE(S) INFORMATION',0)
m = document.add_paragraph()
m.style = document.styles['Normal']

add_df_as_table(df22)

m = document.add_paragraph();m = document.add_paragraph();

c = 0
total_time = 0
for i in my_csv_list:
    c = c + 1
    dfrd = pd.read_csv(i,encoding='windows-1252')
    dfrd = dfrd[dfrd['Output Power (kW)']>25]
    dfrd.insert(0, 'Step Time', 1)    
    total_time = total_time + sum(dfrd['Step Time'])
    print_total_time = 'System/Stack Total Operating Time (output > 25 kW) = '+ str(round(total_time/3600,2))+' hours'

m = document.add_paragraph(print_total_time)
m = document.add_paragraph();m = document.add_paragraph()
m.add_run('Data file(s) associated with this document are available at the following server location: '+ server_location)
document.add_page_break()
###############################PAGE 5+##################################################################################

def file_data_analyzer(list_of_csvs):
    c = 0
    total_time = 0
    for i in list_of_csvs:
        c = c + 1
        m = document.add_paragraph(); m = document.add_paragraph()

        document.add_paragraph('File Name = '+df.iloc[c-1,2])
        document.add_paragraph('File Notes = '+str(df.iloc[c-1,4]))
        
        dfrd = pd.read_csv(i,encoding='windows-1252')
        dfrd = dfrd[dfrd['Output Power (kW)']>25]
        dfrd2 = dfrd.copy()
        dfrd.insert(0, 'Step Time', 1)
        df_analyzed = analyze_data(dfrd)

        add_df_as_table(df_analyzed)
        m = document.add_paragraph()
        
        for g in variable_list:
            document.add_paragraph(g)
            add_df_as_table(df_statistics(dfrd,g))
            m = document.add_paragraph()
        
        m = document.add_paragraph('Cell CVM Data')
        add_df_as_table(cvm_statistics(dfrd))
        m = document.add_paragraph()
        
        m = document.add_paragraph()
        document.add_page_break()
        
        #document.add_heading('DATA FILE '+str(c)+' GRAPHICAL RESULTS',0)
        m = document.add_paragraph(); m = document.add_paragraph()
        
        img = cvm_plotter(dfrd)
        document.add_picture(img, width=Inches(6.5))
        
        img = variable_doubleplot(dfrd,'Time[s]','Output Power (kW)','Power SP (kW)','xlim',210,210,'m','g',28,33,'line')
        document.add_picture(img, width=Inches(6.5))
       
        img = variable_plot(dfrd,'Time[s]','Water SP (%)','xlim','ylim','b',28,'line')
        document.add_picture(img, width=Inches(6.5))
        
        img = variable_plot(dfrd,'Time[s]','Secondary Pressure (bar)','xlim',1.5,'b',28,'line')
        document.add_picture(img, width=Inches(6.5))

        img = variable_plot(dfrd,'Time[s]','Inlet Pressure (bar)','xlim',2.0,'b',28,'line')
        document.add_picture(img, width=Inches(6.5))
        
        img = power_boxplot(dfrd2)
        document.add_picture(img, width=Inches(6.5))

        document.add_page_break()

file_data_analyzer(my_csv_list)
###############TEST PHOTOS#############################

#document.add_heading('TEST PHOTOS',0)
document.add_paragraph()

#photo_list_png = files_location+str("/*.png");my_photo_list_png = (glob.glob(photo_list_png))
#photo_list_jpg = files_location+str("/*.jpg");my_photo_list_jpg = (glob.glob(photo_list_jpg))

c = 0
for s in my_photo_list_png:
    y = format(os.path.basename(my_photo_list_png[c]))
    document.add_paragraph(y)
    document.add_picture(s, width = Inches(6.5))
    document.add_paragraph()
    c = c+1

c = 0
for s in my_photo_list_jpg:
    y = format(os.path.basename(my_photo_list_jpg[c]))
    document.add_paragraph(y)
    document.add_picture(s, width = Inches(6.5))
    document.add_paragraph()
    c = c+1


###############GENERATE FILE###########################
document.save(r'Fuel_Cell_Report.docx')